# Paper demo: Table 8 GPT-2 intervention

This notebook runs the GPT-2 Small generation intervention reported in Table 8. The paper identifies layer 7 component C317 with a weekend-activity pattern and applies an additive offset of -20 at the final prompt token only. All generation is greedy.

In [ ]:
%pip install -q icalens==0.4.0

In [ ]:
from icalens import ICALens

lens = ICALens.from_pretrained(
    "sida/icalens-gpt2-small-pile10k",
)
prompt = "My favorite thing to do on the weekend is to"
layer = 7
component = 317

In [ ]:
profile = lens.component_profile(layer=layer, component=component)
tail = profile["tail_direction"]
print("Selected semantic tail:", tail)
print("Skewness:", round(profile["score_statistics"]["skewness"], 2))
print("Representative corpus occurrences:")
for item in profile["examples"][tail]["occurrences"][:5]:
    print(f"  {item['score']:+.2f}  {item['context']}")

In [ ]:
baseline = lens.generate(
    prompt,
    document_framing="model",
    max_new_tokens=10,
)
print("Baseline:", prompt + baseline)

In [ ]:
steered = lens.generate(
    prompt,
    layer=layer,
    initial_steer=(component, -20.0),
    steer=(component, 0.0),
    steering_scope="current-position",
    document_framing="model",
    max_new_tokens=10,
)
print("Steered:", prompt + steered)

## Reported result

In the paper, the baseline continues with `go to a local bar and have a drink.` while the intervention continues with `play with the kids.` Exact decoding can vary if the model, artifact, Transformers version, or prompt changes.